<a href="https://colab.research.google.com/github/senudidinaya/Dinaya---EfficientNet-B0/blob/Dinaya's-EfficientNet-B0/Dinaya_EfficientNet-B0/notebooks/Dinaya's_Component_%E2%80%94_EfficientNet_B0_(Chest_X_ray_NORMAL_vs_PNEUMONIA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install timm torchmetrics scikit-learn albumentations opencv-python pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# COPY with quotes because the folder name has a space
!cp "/content/drive/MyDrive/DL_Assignment/kaggle.json" /content/kaggle.json

# sanity check: the file must show {"username": "...", "key": "..."}
!ls -l /content/kaggle.json
!cat /content/kaggle.json


In [ ]:
# if kaggle.json is already in /content (I see it on the left), copy it to ~/.kaggle
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/DL_Assignment/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# install kaggle CLI
!pip -q install kaggle

# download the Chest X-ray dataset zip to /content
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content


In [ ]:
# unzip into /content/chest_xray
!unzip -q -o /content/chest-xray-pneumonia.zip -d /content

# list the folder to confirm contents
!ls -la /content/chest_xray

In [ ]:
CFG = {
    "data_root": "/content/chest_xray",
    "img_size": 224,
    "batch_size": 32,
    "epochs": 12,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 10,              # early stop
    "model": "efficientnet_b0",
    "ckpt": "/content/efficientnet_b0_best.pt"
}

import torch
print("CUDA available?", torch.cuda.is_available())


In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]
to3 = transforms.Lambda(lambda img: img.convert("RGB"))

def build_loaders(root, img_size, bs, augment=True):
    if augment:
        train_tf = transforms.Compose([
            to3,
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(7),
            transforms.ColorJitter(0.08,0.08,0.08,0.03),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        ])
    else:
        train_tf = transforms.Compose([
            to3, transforms.Resize((img_size, img_size)),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        ])

    test_tf = transforms.Compose([
        to3, transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

    tr = datasets.ImageFolder(f"{root}/train", train_tf)
    va = datasets.ImageFolder(f"{root}/val",   test_tf)
    te = datasets.ImageFolder(f"{root}/test",  test_tf)

    return (
        DataLoader(tr, batch_size=bs, shuffle=True,  num_workers=4, pin_memory=True),
        DataLoader(va, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True),
        DataLoader(te, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True),
    )

train_loader, val_loader, test_loader = build_loaders(CFG["data_root"], CFG["img_size"], CFG["batch_size"], augment=True)
len(train_loader), len(val_loader), len(test_loader)


In [ ]:
import torch, torch.nn.functional as F
import timm
from torchmetrics.classification import BinaryAUROC

device = "cuda" if torch.cuda.is_available() else "cpu"

# model
model = timm.create_model(CFG["model"], pretrained=True, num_classes=2).to(device)
print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)

# optim + scheduler + metric
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
auroc = BinaryAUROC().to(device)

best_auc, best_state, bad = 0.0, None, 0

for ep in range(CFG["epochs"]):
    # --- train ---
    model.train()
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        out = model(x)
        loss = F.cross_entropy(out, y)
        opt.zero_grad(); loss.backward(); opt.step()
    sch.step()

    # --- validate (ROC-AUC) ---
    model.eval(); auroc.reset()
    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device), y.to(device)
            p1 = model(x).softmax(1)[:,1]
            auroc.update(p1, y)
    vauc = float(auroc.compute().item())
    print(f"epoch {ep+1:02d}/{CFG['epochs']}: val ROC-AUC = {vauc:.4f}")

    # early stop
    if vauc > best_auc:
        best_auc, best_state, bad = vauc, model.state_dict(), 0
    else:
        bad += 1
        if bad > CFG["patience"]:
            print("early stop")
            break

# save best
model.load_state_dict(best_state)
torch.save({"state_dict": model.state_dict(), "best_val_auc": best_auc}, CFG["ckpt"])
print("Best val ROC-AUC:", best_auc)


In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, classification_report
)
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

all_probs, all_preds, all_targets = [], [], []

with torch.no_grad():
    for x,y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()  # P(pneumonia)
        preds = (probs >= 0.5).astype(int)
        all_probs.append(probs)
        all_preds.append(preds)
        all_targets.append(y.numpy())

y_prob = np.concatenate(all_probs)
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_targets)

acc  = accuracy_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)                  # binary F1 (positive=pneumonia)
f1m  = f1_score(y_true, y_pred, average='macro') # macro-F1
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)              # sensitivity/recall
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
spec = tn / (tn + fp + 1e-12)                    # specificity
auc  = roc_auc_score(y_true, y_prob)

print(f"TEST — Acc: {acc:.4f} | F1: {f1:.4f} | Macro-F1: {f1m:.4f} | "
      f"Sensitivity: {rec:.4f} | Specificity: {spec:.4f} | ROC-AUC: {auc:.4f}")

print("\nClassification report:\n",
      classification_report(y_true, y_pred, target_names=['NORMAL','PNEUMONIA']))

# Confusion matrix plot
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots()
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['NORMAL','PNEUMONIA'])
ax.set_yticklabels(['NORMAL','PNEUMONIA'])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_title("Confusion Matrix — Test")
plt.colorbar(im); plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve

# collect val probabilities/targets
model.eval()
val_probs, val_targets = [], []
with torch.no_grad():
    for x,y in val_loader:
        x = x.to(device)
        p1 = torch.softmax(model(x), dim=1)[:,1].detach().cpu().numpy()
        val_probs.append(p1); val_targets.append(y.numpy())
val_prob = np.concatenate(val_probs)
val_true = np.concatenate(val_targets)

# ROC + best threshold (Youden's J)
fpr, tpr, thr = roc_curve(val_true, val_prob)
youden = tpr - fpr
best_idx = youden.argmax()
best_thr = thr[best_idx]
print(f"Best threshold (Youden): {best_thr:.3f}  | Sens={tpr[best_idx]:.4f} Spec={1-fpr[best_idx]:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score, classification_report

# test probs again (you already have y_prob, y_true, but recompute if needed)
test_probs, test_targets = [], []
with torch.no_grad():
    for x,y in test_loader:
        x = x.to(device)
        p1 = torch.softmax(model(x), dim=1)[:,1].detach().cpu().numpy()
        test_probs.append(p1); test_targets.append(y.numpy())
y_prob = np.concatenate(test_probs)
y_true = np.concatenate(test_targets)

y_pred = (y_prob >= best_thr).astype(int)

acc  = accuracy_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
f1m  = f1_score(y_true, y_pred, average='macro')
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)               # sensitivity
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
spec = tn / (tn + fp + 1e-12)
auc  = roc_auc_score(y_true, y_prob)

print(f"[Threshold {best_thr:.3f}] TEST — Acc: {acc:.4f} | F1: {f1:.4f} | Macro-F1: {f1m:.4f} | "
      f"Sens: {rec:.4f} | Spec: {spec:.4f} | ROC-AUC: {auc:.4f}")
print("\nReport:\n", classification_report(y_true, y_pred, target_names=['NORMAL','PNEUMONIA']))